# Building a Text Generation Model with Python

This notebook walks through a full pipeline for training a simple text generation model.  

**Sections:**
1. Data Collection
2. Data Preprocessing
3. Model Design
4. Model Training
5. Text Generation
6. Model Evaluation

Libraries used include PyTorch, NLTK, and requests/BeautifulSoup for optional scraping.

## 1. Data Collection

Collect text data from sources like Project Gutenberg or web scraping using `requests` and `BeautifulSoup`.

```python
import requests

# example: download "Alice in Wonderland"
url = "https://www.gutenberg.org/files/11/11-0.txt"
data = requests.get(url).text

with open("dataset.txt","w",encoding="utf-8") as f:
    f.write(data)
```

```python
from bs4 import BeautifulSoup
import requests

url = "https://example.com/article"
page = requests.get(url)
soup = BeautifulSoup(page.text, "html.parser")
text = soup.get_text()
```

# or use the helper from `dataset.data_collection`
```python
from dataset.data_collection import fetch_and_save

# download plain text directly into dataset.txt
fetch_and_save("https://www.gutenberg.org/files/11/11-0.txt", "dataset.txt")

# scrape selected HTML elements and save result
fetch_and_save(
    "https://example.com/article",
    "dataset.txt",
    scrape=True,
    selectors=["p", "h1"],
)
```

## 2. Data Preprocessing

Clean and tokenize the text data by removing punctuation and converting to lowercase.

```python
import nltk
import string
from collections import Counter
import numpy as np

# download punkt tokenizer
nltk.download('punkt')

# load the raw text
with open("dataset.txt") as f:
    text = f.read().lower()

# tokenize into words
tokens = nltk.word_tokenize(text)

# remove punctuation
tokens = [word for word in tokens if word not in string.punctuation]

print(f"Total tokens: {len(tokens)}")
print(f"Sample tokens: {tokens[:50]}")

# Build vocabulary
word_counts = Counter(tokens)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
vocab_size = len(vocab)

# Create word to index mapping
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print(f"Vocabulary size: {vocab_size}")

# Create sequences for training
seq_length = 50
sequences = []
targets = []

indexed_tokens = [word_to_idx[word] for word in tokens]

for i in range(len(indexed_tokens) - seq_length):
    seq = indexed_tokens[i:i + seq_length]
    target = indexed_tokens[i + seq_length]
    sequences.append(seq)
    targets.append(target)

sequences = np.array(sequences)
targets = np.array(targets)

print(f"Number of sequences: {len(sequences)}")
```

Once tokenized and cleaned, you can:
- Build a vocabulary mapping words to indices
- Create sequences for training
- Save tokens or vocabulary for later use

## 3. Model Design

Define a Recurrent Neural Network (RNN) model with embedding, LSTM, and dense layers using PyTorch.

```python
import torch
import torch.nn as nn

class TextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(TextGenerator, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

# initialize model (later after vocab is built)
# model = TextGenerator(vocab_size=len(vocab), embed_size=128, hidden_size=256)
```

## 4. Model Training

Train the RNN model on the preprocessed sequences using PyTorch's DataLoader and optimizer.

```python
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Assuming vocab, word_to_idx, idx_to_word are defined from preprocessing
# And sequences, targets from earlier

# Hyperparameters
seq_length = 50
batch_size = 64
embed_size = 128
hidden_size = 256
num_epochs = 10
learning_rate = 0.001

# Create data loader
dataset = TensorDataset(torch.tensor(sequences, dtype=torch.long),
                       torch.tensor(targets, dtype=torch.long))
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = TextGenerator(vocab_size=len(vocab), embed_size=embed_size, hidden_size=hidden_size)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        # Get the last output for each sequence
        outputs = outputs[:, -1, :]  # Shape: (batch_size, vocab_size)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}')

print("Training completed!")
```

## 5. Text Generation

Generate text by predicting the next word iteratively based on a starting prompt.

```python
def generate_text(model, start_word, length=20):
    model.eval()
    words = [start_word]
    for i in range(length):
        idx = torch.tensor([[word_to_idx[words[-1]]]])
        output = model(idx)
        next_word_id = torch.argmax(output[0, -1]).item()
        next_word = idx_to_word[next_word_id]
        words.append(next_word)
    return " ".join(words)

# example:
# print(generate_text(model, "Alice"))
```

## 6. Model Evaluation

Two evaluation approaches are used.

### Quantitative Evaluation

Perplexity is used to measure model performance.

Lower perplexity indicates better prediction capability.

### Qualitative Evaluation

Generated sentences are manually evaluated based on:

- Grammar
- Coherence
- Creativity
- Logical flow

### Improvements and Next Steps

- Replace RNN with LSTM or Transformer for better long-term context.
- Consider using Hugging Face Transformers or GPT-style models for state-of-the-art generation.
- Add data loaders, batching, and checkpointing for larger datasets.

**Final System Overview**
1. Collect text data
2. Clean and tokenize it
3. Design the model
4. Train the RNN model
5. Generate sentences from prompts
6. Evaluate model quality using perplexity and human review

✅ Expected outcome example:

Prompt: *Artificial Intelligence is*

Generated text:
"Artificial Intelligence is rapidly changing the way humans interact with technology and data."